# LoRA SFT on a Hugging Face GPU Job

## What you set up (only three things to remember)

1. **`GITHUB_PUBLIC_CLONE_URL`** + **`GIT_CLONE_BRANCH`** — Single branch clone: `git clone --depth 1 --single-branch --branch <branch> <url>`. Set the branch to the one you are developing (not necessarily `main`).
2. **Dataset JSONL on that branch** — Default: `datagen_sft/qwen25_finetune_data.jsonl` (path relative to repo root). The Job only sees files **committed** on `GIT_CLONE_BRANCH`. Regenerate or copy data before pushing. See [README.md](./README.md) in this repo (or your fork on GitHub): `datagen_sft/simulate_and_format.py` → `qwen25_finetune_data.jsonl`.
3. **Hugging Face login** — For starting the paid Job and for **pushing the LoRA adapter** to the Hub.

**No OpenEnv:** This notebook runs supervised fine-tuning only via `datagen_sft/train_qwen_lora.py`. You do **not** need a Space URL or `OPENENV_BASE_URL`.

Use a **`HUB_MODEL_ID` different from your GRPO repo** so LoRA and GRPO adapters are not overwritten.

Default GPU in the config cell: **L40S (48GB)**. For tighter VRAM, set **`BASE_QUANT = "4bit"`** in the config cell. You need HF Pro/Team for Jobs and credits.

In [ ]:
%pip install -q -U "huggingface_hub>=0.28.0"

In [ ]:
import os
import time

from huggingface_hub import get_token, login, run_job, inspect_job, fetch_job_logs

## 1. Login

Uses `HF_TOKEN` from the environment if set (e.g. Colab secrets / GitHub Actions). Otherwise opens the browser or use `getpass` below.

In [ ]:
if not get_token():
    login(add_to_git_credential=False)
else:
    print("Already logged in (HF_TOKEN / cache).")

## 2. Job configuration

Edit **`GITHUB_PUBLIC_CLONE_URL`** and **`GIT_CLONE_BRANCH`**. Ensure **`DATASET_REL_PATH`** points to a JSONL that exists on that branch. Set **`HUB_MODEL_ID`** to a Hub repo name that does not collide with your GRPO run. Then run the launch cell below.

In [ ]:
# Public clone URL. Default matches https://github.com/radharamanaa/OpenEnv-Learn-handwriting (learn_handwriting/ at repo root)
GITHUB_PUBLIC_CLONE_URL = "https://github.com/radharamanaa/OpenEnv-Learn-handwriting.git"
GIT_CLONE_BRANCH = "stage2/long_planning_english"  # must match the branch you pushed to origin (see: git branch --show-current)

HF_USERNAME = "abhijeetmishra101"  # Hugging Face org/user for where the adapter is pushed

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# Distinct from GRPO so adapters are not overwritten
HUB_MODEL_ID = f"{HF_USERNAME}/Qwen2.5-7B-Handwriting-SFT-LoRA"

DATASET_REL_PATH = "datagen_sft/qwen25_finetune_data.jsonl"
OUTPUT_REL_PATH = "outputs/qwen25-handwriting-lora"

# QLoRA: "8bit" (default on CUDA in script), "4bit" (tighter VRAM), "none" (full weights; needs large GPU)
BASE_QUANT = "8bit"
MAX_SEQ_LENGTH = 2048

NUM_TRAIN_EPOCHS = 3.0
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
SAVE_STEPS = 500
LOGGING_STEPS = 10
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

JOB_IMAGE = "pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel"
JOB_FLAVOR = "l40sx1"
JOB_TIMEOUT = "8h"

## 3. Launch training Job

Clones **only** `GIT_CLONE_BRANCH` with `--single-branch` (not necessarily `main`).  
Passes **`HF_TOKEN`** so training can **push the LoRA adapter to Hugging Face**.

In [ ]:
token = get_token()
if not token:
    raise RuntimeError("No HF token. Run the login cell or set HF_TOKEN.")
if "OWNER/REPO" in GITHUB_PUBLIC_CLONE_URL or GITHUB_PUBLIC_CLONE_URL.count("github.com") == 0:
    raise ValueError(
        'Set GITHUB_PUBLIC_CLONE_URL in the config cell to your *public* repo, e.g. https://github.com/yourname/yourname.git (copy from the green Code button on GitHub)'
    )
print("Will git clone:", GITHUB_PUBLIC_CLONE_URL, "branch:", GIT_CLONE_BRANCH)
print("Dataset (on branch):", DATASET_REL_PATH)

remote_script = f"""
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq && apt-get install -y -qq --no-install-recommends git ca-certificates
export PIP_DISABLE_PIP_VERSION_CHECK=1
pip install -q -U pip wheel setuptools
git clone --depth 1 --single-branch --branch "${{GIT_CLONE_BRANCH}}" "${{GITHUB_PUBLIC_CLONE_URL}}" /tmp/lh
REPO_ROOT=/tmp/lh/learn_handwriting
if [ ! -f "$REPO_ROOT/datagen_sft/train_qwen_lora.py" ]; then
  echo "ERROR: expected $REPO_ROOT/datagen_sft/train_qwen_lora.py. We use: git clone ... /tmp/lh, so the repo root is /tmp/lh (not /tmp/lh/Repo-Name). Put learn_handwriting/ at the root of the GitHub repo." >&2
  ls -la /tmp/lh >&2 || true
  exit 1
fi
cd "$REPO_ROOT"
export PIP_ROOT_USER_ACTION=ignore
pip install -q -e "."
pip install -q "torch" "transformers>=4.44.0" "trl>=1.0.0,<2.0.0" "peft" "accelerate" "datasets" "bitsandbytes" "python-dotenv"
export PYTHONPATH="$REPO_ROOT"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python datagen_sft/train_qwen_lora.py \\
  --dataset "$REPO_ROOT/${{DATASET_REL_PATH}}" \\
  --output_dir "$REPO_ROOT/${{OUTPUT_REL_PATH}}" \\
  --model_name "${{MODEL_NAME}}" \\
  --hub_model_id "${{HUB_MODEL_ID}}" \\
  --base_quant "${{BASE_QUANT}}" \\
  --max_seq_length "${{MAX_SEQ_LENGTH}}" \\
  --num_train_epochs "${{NUM_TRAIN_EPOCHS}}" \\
  --per_device_train_batch_size "${{PER_DEVICE_TRAIN_BATCH_SIZE}}" \\
  --gradient_accumulation_steps "${{GRADIENT_ACCUMULATION_STEPS}}" \\
  --learning_rate "${{LEARNING_RATE}}" \\
  --save_steps "${{SAVE_STEPS}}" \\
  --logging_steps "${{LOGGING_STEPS}}" \\
  --lora_r "${{LORA_R}}" \\
  --lora_alpha "${{LORA_ALPHA}}" \\
  --lora_dropout "${{LORA_DROPOUT}}"
echo "Training job script finished."
"""

_job_env = {
    "GITHUB_PUBLIC_CLONE_URL": GITHUB_PUBLIC_CLONE_URL,
    "GIT_CLONE_BRANCH": GIT_CLONE_BRANCH,
    "HUB_MODEL_ID": HUB_MODEL_ID,
    "MODEL_NAME": MODEL_NAME,
    "DATASET_REL_PATH": DATASET_REL_PATH,
    "OUTPUT_REL_PATH": OUTPUT_REL_PATH,
    "BASE_QUANT": BASE_QUANT,
    "MAX_SEQ_LENGTH": str(MAX_SEQ_LENGTH),
    "NUM_TRAIN_EPOCHS": str(NUM_TRAIN_EPOCHS),
    "PER_DEVICE_TRAIN_BATCH_SIZE": str(PER_DEVICE_TRAIN_BATCH_SIZE),
    "GRADIENT_ACCUMULATION_STEPS": str(GRADIENT_ACCUMULATION_STEPS),
    "LEARNING_RATE": str(LEARNING_RATE),
    "SAVE_STEPS": str(SAVE_STEPS),
    "LOGGING_STEPS": str(LOGGING_STEPS),
    "LORA_R": str(LORA_R),
    "LORA_ALPHA": str(LORA_ALPHA),
    "LORA_DROPOUT": str(LORA_DROPOUT),
}

job = run_job(
    image=JOB_IMAGE,
    command=["bash", "-c", remote_script],
    flavor=JOB_FLAVOR,
    timeout=JOB_TIMEOUT,
    secrets={"HF_TOKEN": token},
    env=_job_env,
)

print("Job:", job)
if getattr(job, "url", None):
    print("URL:", job.url)

## 4. (Optional) Stream logs

Poll status and print Job stdout/stderr. Logs are normal **training** output (loss, device discovery, etc.); there is no OpenEnv WebSocket trace in this SFT-only run.

In [ ]:
job_id = getattr(job, "id", None)
if job_id:
    for _ in range(60):
        info = inspect_job(job_id=job_id)
        st = info.status
        print(time.strftime("%H:%M:%S"), st)
        if getattr(st, "stage", str(st)) in ("COMPLETED", "ERROR", "CANCELLED"):
            break
        time.sleep(20)
    for line in fetch_job_logs(job_id=job_id):
        print(line, end="")
else:
    print("No job id on result; open the Job URL in the browser for logs.")